In [3]:
import ee
ee.Initialize(project='gentle-operator-308420')

In [2]:
import time
from bhuvan_flood import BhuvanClient, state_config, layer_name
from bhuvan_flood.stitch import stitch_date
from bhuvan_flood.wms_client import covering_tiles, tile_bbox, tiles_for_bbox


# ── Step 1: State config ────────────────────────────────────────────
print('=' * 70)
print('STEP 1 — State config')
print('=' * 70)
cfg = state_config('Kerala')
print(f"  State        : Kerala")
print(f"  Bhuvan code  : {cfg['code']!r}")
print(f"  GAUL name    : {cfg['gaul']!r}")
print(f"  State bbox   : {cfg['bbox']}")
print(f"    width      : {cfg['bbox'][2] - cfg['bbox'][0]:.4f}° "
      f"(~{(cfg['bbox'][2] - cfg['bbox'][0]) * 111:.0f} km)")
print(f"    height     : {cfg['bbox'][3] - cfg['bbox'][1]:.4f}° "
      f"(~{(cfg['bbox'][3] - cfg['bbox'][1]) * 111:.0f} km)")


# ── Step 2: Use state bbox directly (no district polygon) ───────────
print()
print('=' * 70)
print('STEP 2 — AOI = whole-state bbox')
print('=' * 70)
bbox = cfg['bbox']
print(f"  AOI bbox     : {bbox}")
print(f"  No polygon filter — every tile in the covering set will be fetched.")
print(f"  Output will be a bbox-shaped rectangle, not a state-shaped mask.")


# ── Step 3: Tile-grid covering ──────────────────────────────────────
print()
print('=' * 70)
print('STEP 3 — Tile-grid covering at Bhuvan zoom 10')
print('=' * 70)
tx_min, ty_min, tx_max, ty_max = covering_tiles(bbox)
all_tiles = tiles_for_bbox(bbox)
n_tx = tx_max - tx_min + 1
n_ty = ty_max - ty_min + 1
print(f"  Tile-x range    : {tx_min}..{tx_max}  ({n_tx} cols)")
print(f"  Tile-y range    : {ty_min}..{ty_max}  ({n_ty} rows)")
print(f"  Total tiles     : {len(all_tiles)}  (= {n_tx} × {n_ty})")
print(f"  Canvas size     : {n_tx * 256} × {n_ty * 256} px "
      f"(~{n_tx * n_ty * 256 * 256 * 4 / 1e6:.0f} MB in RAM as RGBA)")


# ── Step 4: Show the first/last few tiles so you can verify alignment ─
print()
print('=' * 70)
print('STEP 4 — Sample tiles (first 3 + last 3)')
print('=' * 70)
def show_tile(i, tx, ty):
    b = tile_bbox(tx, ty)
    print(f"    [{i+1:>4}/{len(all_tiles)}] tx={tx} ty={ty}  "
          f"bbox=({b[0]:.4f}, {b[1]:.4f}, {b[2]:.4f}, {b[3]:.4f})")
for i in range(min(3, len(all_tiles))):
    show_tile(i, *all_tiles[i])
if len(all_tiles) > 6:
    print(f"    … ({len(all_tiles) - 6} tiles in the middle) …")
for i in range(max(3, len(all_tiles) - 3), len(all_tiles)):
    show_tile(i, *all_tiles[i])


# ── Step 5: Build the layer name ────────────────────────────────────
print()
print('=' * 70)
print('STEP 5 — Construct the Bhuvan layer name')
print('=' * 70)
DATE = '2018-07-06'   # known Kerala flood
SUFFIX = '_18'  # try '', '_12', '_18' for different hours
layer = layer_name(cfg['code'], DATE, SUFFIX)
print(f"  Target date   : {DATE}")
print(f"  Hour suffix   : {SUFFIX!r}")
print(f"  Layer name    : {layer!r}")
print(f"    Layer naming uses YYYY_DD_MM (day-then-month, not ISO)")


# ── Step 6: Stitch — fire all the WMS requests ──────────────────────
# NOTE: this prints one line per tile (~500 lines for Kerala). To run
# without per-tile output set verbose=False.
print()
print('=' * 70)
print(f"STEP 6 — Fetch {len(all_tiles)} tiles & stitch (verbose=True)")
print('=' * 70)
print(f"  Expected wall-clock: ~{len(all_tiles) * 0.15:.0f}–"
      f"{len(all_tiles) * 0.3:.0f}s at typical Bhuvan latency.")
print()
t0 = time.time()
mask, transform = stitch_date(
    BhuvanClient(),
    layer,
    bbox,
    polygon=None,        # ← no polygon = full bbox
    verbose=True,
)
elapsed = time.time() - t0


# ── Step 7: Summarise the result ────────────────────────────────────
print()
print('=' * 70)
print('STEP 7 — Summary')
print('=' * 70)
flood_px = int(mask.sum())
total_px = int(mask.size)
pct = 100 * flood_px / total_px
print(f"  Mask shape         : {mask.shape}")
print(f"  Mask dtype         : {mask.dtype}")
print(f"  Total pixels       : {total_px:,}")
print(f"  Flood pixels (=1)  : {flood_px:,}  ({pct:.4f}% of bbox area)")
print(f"  Non-flood (=0)     : {total_px - flood_px:,}")
print(f"  Elapsed            : {elapsed:.1f}s")
print(f"  Per-tile latency   : {elapsed / max(len(all_tiles), 1) * 1000:.0f} ms (average)")
print(f"  Transform (gdal)   : {transform}")
print()

if flood_px == 0:
    print("  ⚠ Zero flood pixels — date may have no Bhuvan data over Kerala,")
    print("    or the hour suffix is wrong. Try '', '_12', '_18'.")
else:
    print(f"  ✓ Result looks sensible — {flood_px:,} flood pixels.")
print()

STEP 1 — State config
  State        : Kerala
  Bhuvan code  : 'kl'
  GAUL name    : 'Kerala'
  State bbox   : (74.6661, 8.0973, 77.5962, 12.9908)
    width      : 2.9301° (~325 km)
    height     : 4.8935° (~543 km)

STEP 2 — AOI = whole-state bbox
  AOI bbox     : (74.6661, 8.0973, 77.5962, 12.9908)
  No polygon filter — every tile in the covering set will be fetched.
  Output will be a bbox-shaped rectangle, not a state-shaped mask.

STEP 3 — Tile-grid covering at Bhuvan zoom 10
  Tile-x range    : 1448..1465  (18 cols)
  Tile-y range    : 438..465  (28 rows)
  Total tiles     : 504  (= 18 × 28)
  Canvas size     : 4608 × 7168 px (~132 MB in RAM as RGBA)

STEP 4 — Sample tiles (first 3 + last 3)
    [   1/504] tx=1448 ty=438  bbox=(74.5312, 12.8320, 74.7070, 13.0078)
    [   2/504] tx=1449 ty=438  bbox=(74.7070, 12.8320, 74.8828, 13.0078)
    [   3/504] tx=1450 ty=438  bbox=(74.8828, 12.8320, 75.0586, 13.0078)
    … (498 tiles in the middle) …
    [ 502/504] tx=1463 ty=465  bbox=(77

In [2]:
from bhuvan_flood import download_bhuvan_flood_day

download_bhuvan_flood_day(
    state='Kerala',
    date='2023-07-06',
    output_path='./final_raster/bhuvan_flood_kerala_2023-07-06.tif',
    tile_cache_dir='./tiles',     # every raw 256x256 PNG goes here
    debug=True,
    verbose=True,                  # print every tile URL too
)

STEP 1 — State config
  State           : Kerala
  Bhuvan code     : 'kl'
  GAUL name       : 'Kerala'
  State bbox      : (74.6661, 8.0973, 77.5962, 12.9908)
    width         : 2.9301° (~325 km)
    height        : 4.8935° (~543 km)

STEP 2 — AOI = district polygon
  AOI source        : FAO GAUL level-2: None
  Geometry type     : GeometryCollection
  Sub-geometries    : 102
  Total vertices    : 3531
  Polygon area (°²) : 3.1260
  Buffered bbox     : (74.6661, 8.0973, 77.5962, 12.9908)
  Buffer applied    : 0.05° (~5.6 km)

STEP 3 — Tile-grid covering at Bhuvan zoom 10
  Tile-x range      : 1448..1465  (18 cols)
  Tile-y range      : 438..465  (28 rows)
  Total tiles       : 504  (= 18 × 28)
  Canvas size       : 4608 × 7168 px (~132 MB RGBA per day in RAM)

STEP 4 — Polygon filter
  Tiles before      : 504
  Tiles dropped     : 356 (no polygon overlap)
  Tiles kept        : 148
  Reduction         : 71%

STEP 5 — Sample tiles (first 3 + last 3 of 148 to fetch)
    [   1/148] tx=144

{'state': 'Kerala',
 'district': None,
 'date': '2023-07-06',
 'output_path': 'final_raster\\bhuvan_flood_kerala_2023-07-06.tif',
 'layer_used': 'flood:kl_2023_06_07_18',
 'has_data': True,
 'flood_pixels': 38942,
 'bbox': (74.6661, 8.0973, 77.5962, 12.9908)}

In [7]:


from bhuvan_flood import download_bhuvan_flood_day

download_bhuvan_flood_day(
    state='Kerala',
    district='Ernakulam',
    date='2023-07-06',
    output_path='./final_raster/bhuvan_flood_kerala_ernakulam_2023-07-06.tif',
    tile_cache_dir='./tiles',
    debug=True,
    verbose=True,
)

STEP 1 — State config
  State           : Kerala
  Bhuvan code     : 'kl'
  GAUL name       : 'Kerala'
  State bbox      : (74.6661, 8.0973, 77.5962, 12.9908)
    width         : 2.9301° (~325 km)
    height        : 4.8935° (~543 km)

STEP 2 — AOI = district polygon
  AOI source        : FAO GAUL level-2: 'Ernakulam'
  Geometry type     : GeometryCollection
  Sub-geometries    : 20
  Total vertices    : 598
  Polygon area (°²) : 0.1893
  Buffered bbox     : (76.11353728258558, 9.742560442093335, 76.87726111086204, 10.344329717386564)
  Buffer applied    : 0.05° (~5.6 km)

STEP 3 — Tile-grid covering at Bhuvan zoom 10
  Tile-x range      : 1457..1461  (5 cols)
  Tile-y range      : 453..456  (4 rows)
  Total tiles       : 20  (= 5 × 4)
  Canvas size       : 1280 × 1024 px (~5 MB RGBA per day in RAM)

STEP 4 — Polygon filter
  Tiles before      : 20
  Tiles dropped     : 4 (no polygon overlap)
  Tiles kept        : 16
  Reduction         : 20%

STEP 5 — Sample tiles (first 3 + last 3 of

{'state': 'Kerala',
 'district': 'Ernakulam',
 'date': '2023-07-06',
 'output_path': 'final_raster\\bhuvan_flood_kerala_ernakulam_2023-07-06.tif',
 'layer_used': None,
 'has_data': False,
 'flood_pixels': 231839625,
 'bbox': (76.11353728258558,
  9.742560442093335,
  76.87726111086204,
  10.344329717386564)}

In [8]:
import ee
ee.Initialize(project='gentle-operator-308420')   # district mode needs EE

from bhuvan_flood import download_bhuvan_kharif_stack

download_bhuvan_kharif_stack(
    state='Kerala',
    district='Ernakulam',
    year=2023,
    output_path='./final_raster/bhuvan_kharif_kerala_ernakulam_2023.tif',
    # tile_cache_dir='./tiles',
    debug=True,
    log=True,        # per-day status line during the run
    # verbose=True,  # uncomment if you also want per-tile URLs
)

STEP 1 — State config
  State           : Kerala
  Bhuvan code     : 'kl'
  GAUL name       : 'Kerala'
  State bbox      : (74.6661, 8.0973, 77.5962, 12.9908)
    width         : 2.9301° (~325 km)
    height        : 4.8935° (~543 km)

STEP 2 — AOI = district polygon
  AOI source        : FAO GAUL level-2: 'Ernakulam'
  Geometry type     : GeometryCollection
  Sub-geometries    : 20
  Total vertices    : 598
  Polygon area (°²) : 0.1893
  Buffered bbox     : (76.11353728258558, 9.742560442093335, 76.87726111086204, 10.344329717386564)
  Buffer applied    : 0.05° (~5.6 km)

STEP 3 — Tile-grid covering at Bhuvan zoom 10
  Tile-x range      : 1457..1461  (5 cols)
  Tile-y range      : 453..456  (4 rows)
  Total tiles       : 20  (= 5 × 4)
  Canvas size       : 1280 × 1024 px (~5 MB RGBA per day in RAM)

STEP 4 — Polygon filter
  Tiles before      : 20
  Tiles dropped     : 4 (no polygon overlap)
  Tiles kept        : 16
  Reduction         : 20%

STEP 5 — Sample tiles (first 3 + last 3 of

{'state': 'Kerala',
 'district': 'Ernakulam',
 'year': 2023,
 'output_path': 'final_raster\\bhuvan_kharif_kerala_ernakulam_2023.tif',
 'n_bands': 140,
 'n_days_with_data': 0,
 'days_with_data': [],
 'days_without': ['2023-06-01',
  '2023-06-02',
  '2023-06-03',
  '2023-06-04',
  '2023-06-05',
  '2023-06-06',
  '2023-06-07',
  '2023-06-08',
  '2023-06-09',
  '2023-06-10',
  '2023-06-11',
  '2023-06-12',
  '2023-06-13',
  '2023-06-14',
  '2023-06-15',
  '2023-06-16',
  '2023-06-17',
  '2023-06-18',
  '2023-06-19',
  '2023-06-20',
  '2023-06-21',
  '2023-06-22',
  '2023-06-23',
  '2023-06-24',
  '2023-06-25',
  '2023-06-26',
  '2023-06-27',
  '2023-06-28',
  '2023-06-29',
  '2023-06-30',
  '2023-07-01',
  '2023-07-02',
  '2023-07-03',
  '2023-07-04',
  '2023-07-05',
  '2023-07-06',
  '2023-07-07',
  '2023-07-08',
  '2023-07-09',
  '2023-07-10',
  '2023-07-11',
  '2023-07-12',
  '2023-07-13',
  '2023-07-14',
  '2023-07-15',
  '2023-07-16',
  '2023-07-17',
  '2023-07-18',
  '2023-07-19',
  